# Multi-tool Agent: Planner + Execution Engine

This demo uses Microsoft Agent Framework's **Magentic orchestration** pattern:

- A manager agent acts as the planner: it creates a task ledger, selects specialists, checks progress, and replans when needed.
- Specialist agents act as the execution engine and invoke deterministic Python tools.
- The workflow combines tool results into one final answer.

The sample task prepares a customer-specific cloud support quote. The local tools make the execution easy to inspect and do not modify external systems.

## 1. Install dependencies

Run this cell once, then restart the kernel if Jupyter asks you to.

In [1]:
%pip install agent-framework==1.0.0b251209 python-dotenv azure-ai-projects==2.0.0b2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Load Foundry configuration

Authenticate first with `az login`. The `.env` file must contain `AI_FOUNDRY_PROJECT_ENDPOINT` and `AI_FOUNDRY_DEPLOYMENT_NAME`.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model = os.getenv("AI_FOUNDRY_DEPLOYMENT_NAME")

if not project_endpoint or not model:
    raise ValueError("Set AI_FOUNDRY_PROJECT_ENDPOINT and AI_FOUNDRY_DEPLOYMENT_NAME in .env")

print("Project endpoint:", project_endpoint)
print("Model deployment:", model)

Project endpoint: https://ajay-agent-project111-resource.services.ai.azure.com/api/projects/ajay-agent-project111
Model deployment: ajay-gpt-4o


## 3. Define the execution tools

Agent Framework turns typed Python functions into tools. Descriptive docstrings help the model choose the correct tool and arguments.

In [3]:
PRODUCT_CATALOG = {
    "basic": {"monthly_price": 120.0, "included_hours": 5, "sla": "next business day"},
    "standard": {"monthly_price": 300.0, "included_hours": 15, "sla": "8 hours"},
    "premium": {"monthly_price": 650.0, "included_hours": 40, "sla": "2 hours"},
}

CUSTOMERS = {
    "CUST-100": {"name": "Contoso Retail", "employees": 220, "discount_percent": 10},
    "CUST-200": {"name": "Fabrikam Health", "employees": 850, "discount_percent": 15},
}

def get_product_catalog() -> dict:
    """Return all cloud support plans, prices, included hours, and SLAs."""
    print("TOOL CALL -> get_product_catalog")
    return PRODUCT_CATALOG

def get_customer_profile(customer_id: str) -> dict:
    """Return the customer profile and negotiated discount for a customer ID."""
    print(f"TOOL CALL -> get_customer_profile(customer_id={customer_id!r})")
    return CUSTOMERS.get(customer_id, {"error": f"Unknown customer ID: {customer_id}"})

def calculate_annual_quote(plan_name: str, discount_percent: float = 0) -> dict:
    """Calculate annual list price, discount amount, and final price for a support plan."""
    print(f"TOOL CALL -> calculate_annual_quote(plan_name={plan_name!r}, discount_percent={discount_percent})")
    plan = PRODUCT_CATALOG.get(plan_name.lower())
    if not plan:
        return {"error": f"Unknown plan: {plan_name}"}
    list_price = plan["monthly_price"] * 12
    discount_amount = list_price * discount_percent / 100
    return {
        "plan": plan_name.lower(),
        "annual_list_price": round(list_price, 2),
        "discount_percent": discount_percent,
        "discount_amount": round(discount_amount, 2),
        "annual_final_price": round(list_price - discount_amount, 2),
        "sla": plan["sla"],
        "included_hours_per_month": plan["included_hours"],
    }

def compare_plan_quotes(first_plan: str, second_plan: str, discount_percent: float = 0) -> dict:
    """Compare the annual prices and service levels of two support plans."""
    print(f"TOOL CALL -> compare_plan_quotes(first_plan={first_plan!r}, second_plan={second_plan!r})")
    first = calculate_annual_quote(first_plan, discount_percent)
    second = calculate_annual_quote(second_plan, discount_percent)
    if "error" in first or "error" in second:
        return {"first": first, "second": second}
    return {
        "first": first,
        "second": second,
        "annual_price_difference": round(second["annual_final_price"] - first["annual_final_price"], 2),
    }

## 4. Create planner and specialist agents

Each agent receives its own Foundry conversation. The planner has no business tools; it delegates work to the specialists based on their names, descriptions, instructions, and capabilities.

In [4]:
from agent_framework import ChatAgent
from agent_framework.azure import AzureAIClient
from azure.ai.projects.aio import AIProjectClient
from azure.identity.aio import AzureCliCredential

resources = []

async def create_foundry_agent(
    name: str,
    description: str,
    instructions: str,
    tools=None,
) -> ChatAgent:
    """Create a conversation-scoped Foundry agent and retain resources for cleanup."""
    credential = AzureCliCredential()
    project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
    openai_client = project_client.get_openai_client()
    conversation = await openai_client.conversations.create()
    chat_client = AzureAIClient(
        project_client=project_client,
        conversation_id=conversation.id,
        model_deployment_name=model,
    )
    agent = chat_client.create_agent(
        name=name,
        description=description,
        instructions=instructions,
        tools=tools,
    )
    resources.append((chat_client, project_client, credential))
    print(f"Created {name} with conversation {conversation.id}")
    return agent

In [5]:
manager_agent = await create_foundry_agent(
    name="Quote-Workflow-Manager",
    description="Plans work, delegates tasks, checks progress, and produces the final answer.",
    instructions=(
        "You are the planner and coordinator. Build a concise plan, delegate every factual or "
        "calculation task to the appropriate specialist, verify tool-backed results, and produce "
        "a final recommendation with assumptions clearly stated."
    ),
)

catalog_agent = await create_foundry_agent(
    name="Catalog-and-Customer-Specialist",
    description="Retrieves support plan details and customer profiles.",
    instructions="Use your tools for every catalog or customer fact. Return concise factual results.",
    tools=[get_product_catalog, get_customer_profile],
)

pricing_agent = await create_foundry_agent(
    name="Pricing-Specialist",
    description="Calculates and compares customer-specific support plan quotes.",
    instructions="Use your pricing tools for all arithmetic. Never estimate prices mentally.",
    tools=[calculate_annual_quote, compare_plan_quotes],
)

proposal_agent = await create_foundry_agent(
    name="Proposal-Writer",
    description="Turns verified findings into a concise business recommendation.",
    instructions="Write a clear recommendation using only the verified facts provided by the team.",
)

Created Quote-Workflow-Manager with conversation conv_74ade107f0f1933900ALCZVnb1KwfZJeQDrCnYkvjrmd0JYYHv
Created Catalog-and-Customer-Specialist with conversation conv_3bae1596db635a6500fcU6ntBjvS4IxhVCqW4mOPxqXQKK5PrL
Created Pricing-Specialist with conversation conv_10b292794573b00a009ugEqWHqWRxmPOslN5B9lvj8YSPXCJXK
Created Proposal-Writer with conversation conv_421dfb3f5fb1d82200YRzUriNMKhQEtCSfaryYMhjOm7IHFwOE


## 5. Build the planner/executor workflow

`MagenticBuilder` supplies the execution engine. The standard manager maintains a task ledger, assigns participants, evaluates progress, and can replan after a stall.

In [6]:
from agent_framework import MagenticBuilder, WorkflowOutputEvent

workflow = (
    MagenticBuilder()
    .participants(
        catalog_specialist=catalog_agent,
        pricing_specialist=pricing_agent,
        proposal_writer=proposal_agent,
    )
    .with_standard_manager(
        agent=manager_agent,
        max_round_count=10,
        max_stall_count=2,
    )
    .build()
)

print("Planner/executor workflow created.")

Adding an edge with Executor or AgentProtocol instances directly is not recommended, because workflow instances created from the builder will share the same executor/agent instances. Consider using a registered name for lazy initialization instead.
Adding an edge with Executor or AgentProtocol instances directly is not recommended, because workflow instances created from the builder will share the same executor/agent instances. Consider using a registered name for lazy initialization instead.
Adding an edge with Executor or AgentProtocol instances directly is not recommended, because workflow instances created from the builder will share the same executor/agent instances. Consider using a registered name for lazy initialization instead.
Adding an edge with Executor or AgentProtocol instances directly is not recommended, because workflow instances created from the builder will share the same executor/agent instances. Consider using a registered name for lazy initialization instead.
Addi

Planner/executor workflow created.


## 6. Run and observe delegation

The prompt requires retrieval, calculations, comparison, and writing. Watch the streamed events and `TOOL CALL` lines to see the plan being executed.

In [7]:
task = """
Prepare a support-plan recommendation for customer CUST-100.
"""

final_result = None
async for event in workflow.run_stream(task):
    print(f"{type(event).__name__}: {event}")
    if isinstance(event, WorkflowOutputEvent):
        final_result = event.data

print("\nFINAL RESULT\n", final_result)

WorkflowStartedEvent: WorkflowStartedEvent(origin=WorkflowEventSource.FRAMEWORK, data=None)
WorkflowStatusEvent: WorkflowStatusEvent(state=WorkflowRunState.IN_PROGRESS, data=None, origin=WorkflowEventSource.FRAMEWORK)
ExecutorInvokedEvent: ExecutorInvokedEvent(executor_id=magentic_orchestrator, data=
Prepare a support-plan recommendation for customer CUST-100.
)
AgentRunUpdateEvent: AgentRunUpdateEvent(executor_id=magentic_orchestrator, messages=
Prepare a support-plan recommendation for customer CUST-100.
)
AgentRunUpdateEvent: AgentRunUpdateEvent(executor_id=magentic_orchestrator, messages=
We are working to address the following user request:


Prepare a support-plan recommendation for customer CUST-100.



To answer this request we have assembled the following team:

- catalog_specialist: Retrieves support plan details and customer profiles.
- pricing_specialist: Calculates and compares customer-specific support plan quotes.
- proposal_writer: Turns verified findings into a concise

## 7. Try another task

Change the request to observe how the planner selects a different sequence of specialists and tools.

In [ ]:
###OPTIONAL - Task 

task = "For CUST-200, calculate the annual Premium quote and write a three-bullet executive summary."

async for event in workflow.run_stream(task):
    if isinstance(event, WorkflowOutputEvent):
        print(event.data)

ServiceResponseException: <class 'agent_framework_azure_ai._client.AzureAIClient'> service failed to complete the prompt: Error code: 429 - {'error': {'code': 'rate_limit_exceeded', 'message': 'Model deployment rate limit exceeded. Too Many Requests. To request more quota, see https://learn.microsoft.com/en-us/azure/ai-services/openai/quotas-limits', 'type': 'error', 'additionalInfo': {'request_id': '9be02e1e6b5ea388fa4cd1beb3e2e963', 'rate_limit_source': 'rapi'}}}

## 8. Cleanup

In [9]:
for chat_client, project_client, credential in resources:
    await chat_client.close()
    await project_client.close()
    await credential.close()

resources.clear()
print("Closed Azure clients.")

Closed Azure clients.
